# Разведочный анализ (EDA) синтетических сеансов связи на низкой околоземной орбите (LEO)

**Темы** разведочный анализ данных, оценка качества, очистка, визуализация, ETL (извлечение, преобразование и загрузка) и SQLite (легковесная реляционная база данных).

**Задача** 
Требовалось внимательно изучить табличные данные перед загрузкой в базу. Нужно было найти ошибки (пропуски, дубликаты, физически невозможные значения), понять, какие признаки сильнее всего влияют на объём переданных данных, и оценить вероятность успешного сеанса связи.

**Цель** 
Подготовить данные к безопасной загрузке в реляционную базу данных и подробно описать свойства синтетических сеансов связи.

*(Небольшое пояснение: **LEO** (низкая околоземная орбита) - это орбита, где спутники летают быстро, поэтому сеанс связи с наземной станцией длится всего несколько минут. **Синтетические сеансы** - это не реальная телеметрия с космических аппаратов, а данные, сгенерированные с помощью математической модели для отработки пайплайна. Генератор учитывает упрощенную физику: угол возвышения спутника, расстояние, дождь, шумы. Все выводы относятся только к симуляции).*

Исходный CSV-файл лежит [в Google Drive](https://drive.google.com/file/d/1SakihMBeatb_7KdQSSzXBU_BIjdQpYyl/view) (чтобы не раздувать репозиторий на GitHub). Если локального файла нет, ноутбук сам сгенерирует точно такие же строки с фиксированным seed (зерном случайных чисел).

Сначала проверяются размер, типы данных, пропуски, дубликаты и физические диапазоны. Потом анализируются распределения, корреляции (связи между признаками), сравнения групп и динамика. А в конце запускается реальный ETL-процесс в SQLite.

In [1]:
from pathlib import Path
import json
import sys
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, HTML

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from leo_data_pipeline.generator import FIELDS, generate_dataset
from leo_data_pipeline.transform import transform
from leo_data_pipeline.pipeline import run_pipeline

RAW = ROOT / 'data/raw/leo_passes_2026.csv'
if not RAW.exists():
    generate_dataset(RAW, count=4200, seed=202611)
raw = pd.read_csv(RAW, low_memory=False)
print('Размер raw:', raw.shape)
print('Индекс:', raw.columns.tolist())
assert raw.shape[0] >= 1000 and raw.shape[1] >= 15


Размер raw: (4244, 25)
Индекс: ['event_id', 'timestamp_utc', 'satellite_id', 'orbital_plane', 'station_name', 'station_lat_deg', 'station_lon_deg', 'frequency_ghz', 'elevation_deg', 'slant_range_km', 'pass_duration_s', 'rain_rate_mm_h', 'cloud_cover_pct', 'tx_power_w', 'tx_antenna_gain_dbi', 'rx_antenna_gain_dbi', 'polarization', 'mission_priority', 'system_temperature_k', 'path_loss_db', 'snr_db', 'capacity_mbps', 'transferred_gb', 'link_status', 'operator_note']


## 1. Первичное знакомство со схемой и типами данных

Первым делом оценивается *действительная* картина, а не ожидаемая. Проверяются количество строк, типы данных (`dtypes`), первые строки и уникальность идентификаторов. 

*(Обычно на этом этапе вылезают сюрпризы. Например, если в числах вместо точки использовали запятую, Pandas сделает колонку строковой (`object`), и математику с ней уже не посчитаешь. Или вместо нуля кто-то написал `n/a`, и колонка стала текстовой. Всё это будет исправлено на этапе Transform).*

In [2]:
display(raw.head(5))
print('Типы:')
print(raw.dtypes.to_string())
print('Уникальных event_id:', raw.event_id.nunique(), 'из', len(raw))

,event_id,timestamp_utc,satellite_id,orbital_plane,station_name,station_lat_deg,station_lon_deg,frequency_ghz,elevation_deg,slant_range_km,pass_duration_s,rain_rate_mm_h,cloud_cover_pct,tx_power_w,tx_antenna_gain_dbi,rx_antenna_gain_dbi,polarization,mission_priority,system_temperature_k,path_loss_db,snr_db,capacity_mbps,transferred_gb,link_status,operator_note
0,LEO-001168,2026-09-08T18:24:01Z,KA-10,3,Yakutsk,62.03,129.73,8.4,35.234,1250.728,384,0,38.83,15.429,9.807,31.787,LHCP,1,342.82,172.879,6.568,33.953,1.1520,success,nominal
1,LEO-001465,2026-09-10T17:49:03Z,KA-14,4,Kaliningrad,54.71,20.51,8.2,65.629,745.809,662,0,59.04,21.216,11.850,30.952,LHCP,1,328.24,168.179,14.048,64.934,3.8101,success,NaN
2,LEO-003077,2026-09-21T11:17:47Z,KA-07,2,Kaliningrad,54.71,20.51,8.4,50.208,891.706,495,0,28.45,21.823,9.493,31.230,RHCP,2,327.52,169.940,10.341,48.988,2.2281,success,monitor
3,LEO-000650,2026-09-05T07:39:53Z,KA-01,1,Novosibirsk,55.03,82.92,8.4,42.502,1067.249,502,2.144,NaN,11.426,12.648,30.776,LHCP,3,386.45,171.501,7.652,38.095,2.0817,success,NaN
4,LEO-001226,2026-09-09T03:39:40Z,KA-11,3,Yakutsk,62.03,129.73,8.0,67.046,732.476,636,2.368,42.23,14.354,9.831,28.572,LHCP,1,312.12,167.808,8.211,40.295,2.1085,success,nominal


Типы:
event_id                 object
timestamp_utc            object
satellite_id             object
orbital_plane             int64
station_name             object
station_lat_deg         float64
station_lon_deg         float64
frequency_ghz           float64
elevation_deg           float64
slant_range_km          float64
pass_duration_s           int64
rain_rate_mm_h           object
cloud_cover_pct         float64
tx_power_w              float64
tx_antenna_gain_dbi     float64
rx_antenna_gain_dbi     float64
polarization             object
mission_priority          int64
system_temperature_k    float64
path_loss_db            float64
snr_db                  float64
capacity_mbps           float64
transferred_gb          float64
link_status              object
operator_note            object
Уникальных event_id: 4200 из 4244


## 2. Оценка полноты данных, дубликатов и недостоверных значений

Перед загрузкой в базу обязательно выявляется неполнота и нарушения схемы. Оценивается доля пропусков (missingness) и количество дубликатов по `event_id`. 

*(Генератор специально создал проблемные строки, чтобы было на чём отработать очистку).*

In [3]:
missing = (raw.isna().mean() * 100).sort_values(ascending=False)
q = pd.DataFrame({'missing_pct': missing, 'missing_count': raw.isna().sum().reindex(missing.index)})
display(q.head(12))
print('Дубли по event_id:', raw.duplicated(subset=['event_id']).sum())
print('Недопустимая elevation:', (pd.to_numeric(raw.elevation_deg, errors='coerce') < 0).sum())
print('Сырые значения rain (первые 8):', raw.rain_rate_mm_h.iloc[:8].tolist())

,missing_pct,missing_count
operator_note,65.857681,2795
rain_rate_mm_h,2.097078,89
cloud_cover_pct,1.720075,73
timestamp_utc,0.000000,0
station_name,0.000000,0
station_lat_deg,0.000000,0
satellite_id,0.000000,0
orbital_plane,0.000000,0
event_id,0.000000,0
elevation_deg,0.000000,0


Дубли по event_id: 44
Недопустимая elevation: 23
Сырые значения rain (первые 8): ['0', '0', '0', '2.144', '2.368', '0', '2.207', nan]


## 3. Преобразование данных (Transform) и диагностический отчёт

Здесь применяется функция обработки. Ее задачи:
1. Проверить, чтобы колонки строго совпадали со схемой.
2. Удалить повторные события по ключу `event_id`.
3. Привести числа со строками и запятыми к нормальному `float`, а даты - к UTC (мировому времени).
4. Отвергнуть нефизические значения (например, отрицательную высоту) с подсчетом, а не «подчищать» их обрезкой.
5. Заполнить пропуски осадков и облачности **медианой по конкретной станции** (если для станции данных нет - берется глобальная медиана).
6. Добавить производные признаки (например, класс дождя) и проверить физический предел объёма переданных данных.

In [4]:
clean, quality = transform(raw)
print(json.dumps(quality, ensure_ascii=False, indent=2))
assert clean.event_id.is_unique
assert clean.elevation_deg.between(0, 90).all()
assert clean.rain_rate_mm_h.ge(0).all()
display(clean[['event_id','timestamp_utc','station_name','snr_db','transferred_gb','rain_class']].head())

{
  "raw_rows": 4244,
  "duplicate_event_ids_removed": 44,
  "rejected_schema_domain": 22,
  "rejected_environment": 18,
  "rejected_volume": 0,
  "clean_rows": 4160,
  "column_count_raw": 25,
  "unique_event_ids_ratio": 1.0,
  "initial_missing_by_column": {
    "event_id": 0,
    "timestamp_utc": 0,
    "satellite_id": 0,
    "orbital_plane": 0,
    "station_name": 0,
    "station_lat_deg": 0,
    "station_lon_deg": 0,
    "frequency_ghz": 0,
    "elevation_deg": 0,
    "slant_range_km": 0,
    "pass_duration_s": 0,
    "rain_rate_mm_h": 89,
    "cloud_cover_pct": 73,
    "tx_power_w": 0,
    "tx_antenna_gain_dbi": 0,
    "rx_antenna_gain_dbi": 0,
    "polarization": 0,
    "mission_priority": 0,
    "system_temperature_k": 0,
    "path_loss_db": 0,
    "snr_db": 0,
    "capacity_mbps": 0,
    "transferred_gb": 0,
    "link_status": 0,
    "operator_note": 2795
  },
  "initial_missing_rain_fraction": 0.020970782280867108,
  "environment_missing_imputed": {
    "rain_rate_mm_h": 89,
  

,event_id,timestamp_utc,station_name,snr_db,transferred_gb,rain_class
0,LEO-000002,2026-09-01 00:08:42+00:00,Yakutsk,6.558,1.6819,dry
1,LEO-000003,2026-09-01 00:19:10+00:00,Moscow,12.449,4.7086,heavy
2,LEO-000004,2026-09-01 00:29:11+00:00,Murmansk,6.795,1.0879,heavy
3,LEO-000005,2026-09-01 00:39:18+00:00,Novosibirsk,14.437,4.1631,moderate
4,LEO-000006,2026-09-01 00:47:50+00:00,Moscow,8.796,2.1839,heavy


## 4. Описательные статистики и группировки данных

Для числовых полей рассчитываются среднее, медиана, квартили и экстремумы. Для категорий - частоты. 

*(Важный момент: медиана гораздо устойчивее к выбросам, чем среднее. Стандартное отклонение показывает разброс, но само по себе не говорит о качестве данных).*

In [5]:
num_cols = ['elevation_deg','slant_range_km','rain_rate_mm_h','snr_db','capacity_mbps','transferred_gb']
display(clean[num_cols].describe().T.round(3))
display(clean.groupby('station_name').agg(
    total_contacts=('event_id','size'),
    success_rate=('usable_pass','mean'),
    mean_snr=('snr_db','mean'),
    total_gb=('transferred_gb','sum')
).sort_values('total_gb', ascending=False).round(2))

,count,mean,std,min,25%,50%,75%,max
elevation_deg,4160.0,44.088,21.824,6.024,25.388,44.047,62.786,82.000
slant_range_km,4160.0,1118.081,411.377,537.691,745.519,1041.766,1450.694,1999.033
rain_rate_mm_h,4160.0,1.696,1.754,0.000,0.000,1.228,2.803,9.839
snr_db,4160.0,8.695,4.101,-2.968,5.682,8.700,11.706,19.868
capacity_mbps,4160.0,42.378,17.725,0.000,30.699,42.252,54.768,90.956
transferred_gb,4160.0,2.271,1.463,0.000,1.086,2.028,3.268,7.289


,total_contacts,success_rate,mean_snr,total_gb
station_name,,,,
Vladivostok,711,0.95,8.70,1608.18
Murmansk,697,0.95,8.77,1600.90
Kaliningrad,714,0.95,8.64,1583.13
Moscow,691,0.95,8.71,1582.14
Yakutsk,684,0.95,8.70,1550.65
Novosibirsk,663,0.95,8.65,1522.88


## 5. Статическая визуализация с помощью библиотеки Seaborn (инструмент для графиков)

Построена сетка 2×2. На ней отображены распределение SNR (отношение сигнал/шум), зависимость геометрии (угол и расстояние), временной ряд и матрица корреляций Пирсона. Все оси подписаны, единицы измерения указаны.

*(Корреляция Пирсона показывает, как признаки меняются вместе. Но важно помнить, что корреляция не доказывает причинно-следственную связь).*

In [6]:
sns.set_theme(style='whitegrid', context='notebook', palette=['#164e63','#0891b2','#7c3aed','#c2410c'])
fig, ax = plt.subplots(2, 2, figsize=(14, 9), layout='constrained')
sns.histplot(data=clean, x='snr_db', hue='link_status', bins=35, ax=ax[0,0], alpha=.6)
ax[0,0].set(title='Distribution of SNR', xlabel='SNR (dB)')
sns.scatterplot(data=clean.sample(n=min(850,len(clean)), random_state=2026),
                x='elevation_deg', y='slant_range_km', hue='rain_class', ax=ax[0,1], alpha=.5, s=18)
ax[0,1].set(title='Elevation and distance', xlabel='Elevation (deg)', ylabel='Distance (km)')
daily = clean.groupby('contact_date', as_index=False).transferred_gb.sum()
sns.lineplot(data=daily, x='contact_date', y='transferred_gb', ax=ax[1,0], marker='o')
ax[1,0].tick_params(axis='x', rotation=45)
ax[1,0].set(title='Payload transmitted per UTC date', xlabel='UTC date', ylabel='GB')
sns.heatmap(clean[['snr_db','elevation_deg','slant_range_km','rain_rate_mm_h','capacity_mbps']].corr(),
            annot=True, fmt='.2f', center=0, cmap='BrBG', ax=ax[1,1])
ax[1,1].set_title('Pearson correlation (synthetic)')
plt.show()

/tmp/ipykernel_728/534864798.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Сравнение групп по дождю и отношению сигнал/шум (SNR)

Проверяется гипотеза, заложенная в генератор: при сильном дожде связь обычно ухудшается.

*(Ещё раз подчеркну, что все корреляции здесь - это свойства математической модели, а не оценка того, как осадки влияют на реальные спутники).*

In [7]:
by_rain = clean.groupby('rain_class', observed=True).agg(
    n=('event_id','size'), mean_snr=('snr_db','mean'),
    success_rate=('usable_pass','mean'), mean_volume_gb=('transferred_gb','mean'))
display(by_rain.round(3))
print('Pearson(rain,SNR) =', round(clean[['rain_rate_mm_h','snr_db']].corr().iloc[0,1],3))
print('Spearman(rain,SNR) =', round(clean[['rain_rate_mm_h','snr_db']].corr(method='spearman').iloc[0,1],3))

,n,mean_snr,success_rate,mean_volume_gb
rain_class,,,,
dry,1498,8.903,0.954,2.300
heavy,936,8.182,0.938,2.155
moderate,1726,8.791,0.947,2.310


Pearson(rain,SNR) = -0.07
Spearman(rain,SNR) = -0.07


## 7. Интерактивная визуализация с помощью Plotly Express (библиотека для интерактивных графиков)

Интерактивный график показывает SNR в зависимости от угла возвышения; цвета различают наземные станции, размер точки - переданный объём. Подсказки содержат ID спутника, интенсивность дождя и статус.

*(Важно помнить: GitHub не отображает интерактивные HTML-файлы прямо в браузере. Чтобы посмотреть график, нужно скачать файл `03_interactive_snr.html` из репозитория и открыть его в любом браузере у себя на компьютере).*

In [8]:
sample = clean.sample(n=min(len(clean),900), random_state=2026)
fig = px.scatter(sample, x='elevation_deg', y='snr_db', color='station_name',
                 size='transferred_gb', hover_data=['satellite_id','rain_rate_mm_h','link_status'],
                 title='Interactive synthetic LEO downlink contacts',
                 labels={'elevation_deg':'Elevation (degrees)','snr_db':'Signal-to-noise ratio (dB)',
                         'station_name':'Ground station'},
                 template='plotly_white', opacity=.65)
fig.update_layout(height=500, legend_title_text='Station')
html_path = ROOT/'reports/figures/03_interactive_snr.html'
html_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(html_path, include_plotlyjs='cdn', full_html=True)
display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))
print('Saved interactive HTML:', html_path.relative_to(ROOT))

Saved interactive HTML: reports/figures/03_interactive_snr.html


## 8. Полный цикл ETL (извлечение, преобразование, загрузка) и SQL-валидация

Запускается полный пайплайн (аналог команды `leo-etl run` из терминала). Он создаёт очищенный CSV-файл, базу данных SQLite с двумя таблицами, строит сводки и сохраняет графики.

Затем выполняется проверка в SQLite: сработали ли первичные ключи, нет ли битых внешних ключей, совпадает ли количество строк с отчётом и сохраняется ли свойство идемпотентности (повторный запуск не создает дублей).

In [9]:
report = run_pipeline(input_path=RAW,
                      db_path=ROOT/'data/processed/leo_passes.sqlite',
                      output_path=ROOT/'data/processed/cleaned_contacts.csv',
                      reports_dir=ROOT/'reports')
with sqlite3.connect(ROOT/'data/processed/leo_passes.sqlite') as con:
    assert con.execute('PRAGMA foreign_key_check').fetchall() == []
    n = con.execute('SELECT COUNT(*) FROM contacts').fetchone()[0]
    by_station_sql = pd.read_sql_query("""SELECT station_name, COUNT(*) AS events, ROUND(SUM(transferred_gb),2) AS total_gb
                                   FROM contacts GROUP BY station_name ORDER BY total_gb DESC""", con)
assert n == report['clean_rows']
print('SQL rows:', n)
display(by_station_sql)

SQL rows: 4160


,station_name,events,total_gb
0,Vladivostok,711,1608.18
1,Murmansk,697,1600.90
2,Kaliningrad,714,1583.13
3,Moscow,691,1582.14
4,Yakutsk,684,1550.65
5,Novosibirsk,663,1522.88


## 9. Выводы и ограничения исследования

1. Набор данных соответствует требованиям (25 исходных признаков, больше 1000 строк). Крупные файлы хранятся в Google Drive, а не в репозитории.
2. В сырых данных полно мусора: дубликаты, пропуски, недопустимые физические значения. Без очистки их опасно записывать в реляционную базу.
3. ETL-пайплайн удаляет дубли по `event_id`, приводит типы, отбрасывает неверную физику и заполняет пропуски медианами. Вся статистика потерь сохранена в `quality_report.json`.
4. Успешность сеанса и скорость передачи данных зависят от геометрии (угла), мощности сигнала и погодных условий. *(Но в реальной жизни эти зависимости нужно проверять на настоящей телеметрии, так как модель сильно упрощена).*
5. В SQLite настроены правильные ключи, связи и индексы. Повторный импорт данных не создает дублей.
6. Подготовлены удобные статические визуализации и интерактивный график Plotly. Построенные графики можно использовать для разведки и проверки гипотез, но нельзя применять для оперативного мониторинга реального оборудования.

**Главные ограничения** в данных есть синтетическое смещение (правила генерации придуманы искусственно), модель орбит упрощена. Переносить эти выводы на реальную космонавтику без дополнительной валидации нельзя. Эти данные нельзя выдавать за реальные замеры.